# AITA Coursework – Task 2  
## Structured Information Extraction from Clinical Trial Abstracts

---

This coursework focuses on converting unstructured clinical trial abstracts into structured representations using the EBM-NLP dataset. The goal is to extract key elements of clinical studies—specifically Population, Intervention, and Outcome (PIO)—and organise them into a tabular format.

Structured extraction enables downstream applications such as evidence synthesis, search, and clinical decision support. This task explores how different modelling strategies affect the quality and usability of extracted information.

## Task 1 Summary (Preliminary Clustering Analysis)

As a preliminary step, clustering techniques were applied to sentence-level embeddings of clinical abstracts to investigate whether natural groupings align with schema fields (Population, Intervention, Outcome).

K-means and hierarchical clustering (HAC) were explored using sentence embeddings. However, the resulting clusters did not clearly correspond to distinct PIO categories. Sentences often contained overlapping information (e.g., both population and intervention details), leading to mixed clusters.

### Key Observation
Clustering alone is insufficient for structured extraction in this domain. This motivates the need for more fine-grained approaches, such as token-level sequence tagging.

---

Based on this observation, Task 2 focuses on designing a supervised information extraction pipeline.

## Task 2: Structured Information Extraction

The objective of Task 2 is to design and evaluate an end-to-end pipeline that extracts structured information from clinical trial abstracts.

Given token-level annotations from the EBM-NLP dataset, the goal is to:
- Identify entity spans corresponding to Population, Intervention, and Outcome
- Convert token-level predictions into meaningful text spans
- Organise extracted information into a structured PIO-style table

The quality of extraction is evaluated using both token-level and span-level metrics.

## Design Axis: Training Strategy

This work explores the impact of different training regimes on extraction performance. Specifically, the following strategies are compared:

- **Zero-shot**: Rule-based extraction using predefined cue words (no training)
- **Few-shot**: Supervised learning using a small subset of training data (100 documents)
- **Full-train**: Supervised learning using the complete training dataset

### Hypothesis
- Zero-shot will provide a weak baseline due to lack of learning
- Few-shot will capture useful patterns with limited supervision
- Full-train will improve recall but may introduce noise and false positives

This axis allows us to study how the amount of supervision affects extraction quality and robustness.
---

## Pipeline Overview

The extraction pipeline follows an end-to-end approach:

1. Load token-level data from the EBM-NLP dataset
2. Convert raw annotations into BIO format
3. Extract token-level features (lexical and contextual)
4. Train a classification model (Logistic Regression)
5. Predict BIO labels on test data
6. Convert BIO sequences into text spans
7. Evaluate predictions using span-level metrics
8. Generate structured PIO-style tables

This approach directly maps tokens to entity labels without decomposing the task into intermediate steps.

######################################################################################################################

## Imports

The following libraries are used for data handling, feature extraction, model training, and evaluation.

In [2]:
# importing liabraries

import os
import tarfile
import random
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

## Section 1: Dataset Loading and Extraction

The EBM-NLP dataset is provided as a compressed `.tar.gz` archive.  
This section handles extraction (if required) and prepares the dataset directory for further processing.

To avoid repeated extraction and improve efficiency, the dataset is only extracted if it is not already available locally.

In [19]:
tar_path = r"C:\Users\vikra\OneDrive\Desktop\Study_Board\Bristol\SEM 2\Introduction to AI and Text Analytics\AITA Coursework\EBM-NLP-master\ebm_nlp_2_00.tar.gz"

extract_root = os.path.dirname(tar_path)
base_folder_name = "ebm_nlp_2_00"
base_path = os.path.join(extract_root, base_folder_name)

# normalize for printing / matching
base_path = base_path.replace("\\", "/")
tar_path = tar_path.replace("\\", "/")


In [20]:
# extract if not exists
if not os.path.exists(base_path):
    print("Dataset folder not found. Extracting from .tar.gz ...")

    if tarfile.is_tarfile(tar_path):
        with tarfile.open(tar_path, "r:gz") as f:
            f.extractall(path=extract_root)

        print("Extraction completed.")
    else:
        raise ValueError("Provided file is not a valid .tar.gz archive")
else:
    print("Dataset already extracted. Skipping extraction.")

print("Using dataset folder:", base_path)

Dataset already extracted. Skipping extraction.
Using dataset folder: C:/Users/vikra/OneDrive/Desktop/Study_Board/Bristol/SEM 2/Introduction to AI and Text Analytics/AITA Coursework/EBM-NLP-master/ebm_nlp_2_00


In [22]:
all_files = []

for root, dirs, files in os.walk(base_path):
    for file in files:
        full_path = os.path.join(root, file).replace("\\", "/")
        all_files.append(full_path)

print("Total files found:", len(all_files))

Total files found: 462136


---

## Section 2. Discovering Files and Filtering

The EBM-NLP dataset contains:
- Tokenised documents (`.tokens`)
- Annotation files for each entity type:
  - Participants
  - Interventions
  - Outcomes

Each entity is further divided into **train** and **test** splits.

In this section, we:
- Filter all token files
- Separate annotation files for each entity type (P, I, O)
- Maintain train/test splits for supervised learning
- Combine train and test sets where required for flexible access

This structured organisation enables efficient loading and processing of data in subsequent steps.

In [23]:
token_files = [f for f in all_files if f.endswith(".tokens")]

participant_train_files = [
    f for f in all_files
    if "annotations/aggregated/hierarchical_labels/participants/train/" in f
    and f.endswith(".AGGREGATED.ann")
]

participant_test_files = [
    f for f in all_files
    if "annotations/aggregated/hierarchical_labels/participants/test/" in f
    and f.endswith(".AGGREGATED.ann")
]

intervention_train_files = [
    f for f in all_files
    if "annotations/aggregated/hierarchical_labels/interventions/train/" in f
    and f.endswith(".AGGREGATED.ann")
]

intervention_test_files = [
    f for f in all_files
    if "annotations/aggregated/hierarchical_labels/interventions/test/" in f
    and f.endswith(".AGGREGATED.ann")
]

outcome_train_files = [
    f for f in all_files
    if "annotations/aggregated/hierarchical_labels/outcomes/train/" in f
    and f.endswith(".AGGREGATED.ann")
]

outcome_test_files = [
    f for f in all_files
    if "annotations/aggregated/hierarchical_labels/outcomes/test/" in f
    and f.endswith(".AGGREGATED.ann")
]

all_participant_files = participant_train_files + participant_test_files
all_intervention_files = intervention_train_files + intervention_test_files
all_outcome_files = outcome_train_files + outcome_test_files

print("\nFiltered file counts:")
print("Total token files:", len(token_files))
print("Participants train files:", len(participant_train_files))
print("Participants test files:", len(participant_test_files))
print("Interventions train files:", len(intervention_train_files))
print("Interventions test files:", len(intervention_test_files))
print("Outcomes train files:", len(outcome_train_files))
print("Outcomes test files:", len(outcome_test_files))


Filtered file counts:
Total token files: 9986
Participants train files: 9218
Participants test files: 760
Interventions train files: 9492
Interventions test files: 756
Outcomes train files: 9362
Outcomes test files: 762


--------------------------------------------------------------------------------------------------------------------------------

## Section 3. Loading Data and Document Construction

## Data Loading and Document Construction

In this section, the filtered token and annotation files are loaded and combined into structured document-level representations.

Each document is represented as a dictionary containing:
- `doc_id`: unique identifier of the abstract
- `tokens`: list of tokenised words
- `labels`: corresponding annotation labels

The process involves:
1. Reading token and annotation files
2. Matching documents using their IDs
3. Ensuring token-label alignment
4. Constructing structured datasets for each entity type (Participants, Interventions, Outcomes)

Separate datasets are maintained for training and testing to support supervised learning and evaluation.

A sanity check is performed to verify that token and label sequences are correctly aligned.

In [26]:
def read_lines_from_file(file_path, as_int=False):
    """
    Reads a file and returns non-empty lines.

    Parameters:
    - file_path: path to the file
    - as_int (bool): if True, converts each line to integer (used for label files)

    Returns:
    - List of strings (tokens) or integers (labels)
    """
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = [line.strip() for line in f if line.strip()]

    if as_int:
        return [int(x) for x in lines]

    return lines

In [25]:
## Sanity check

sample_token_file = random.choice(token_files)
filename = sample_token_file.split("/")[-1].replace(".tokens", "")

sample_participant_files = [
    f for f in all_participant_files
    if f.endswith(f"{filename}.AGGREGATED.ann")
]

sample_intervention_files = [
    f for f in all_intervention_files
    if f.endswith(f"{filename}.AGGREGATED.ann")
]

sample_outcome_files = [
    f for f in all_outcome_files
    if f.endswith(f"{filename}.AGGREGATED.ann")
]

sample_participant_file = random.choice(sample_participant_files) if sample_participant_files else None
sample_intervention_file = random.choice(sample_intervention_files) if sample_intervention_files else None
sample_outcome_file = random.choice(sample_outcome_files) if sample_outcome_files else None

print("\nMatched sample files:")
print("Token:", sample_token_file)
print("Participant:", sample_participant_file)
print("Intervention:", sample_intervention_file)
print("Outcome:", sample_outcome_file)

tokens = read_lines_from_file(sample_token_file)
p_labels = read_lines_from_file(sample_participant_file, as_int=True) if sample_participant_file else None
i_labels = read_lines_from_file(sample_intervention_file, as_int=True) if sample_intervention_file else None
o_labels = read_lines_from_file(sample_outcome_file, as_int=True) if sample_outcome_file else None

print("\nLength checks:")
print("P aligned with tokens:", len(tokens) == len(p_labels) if p_labels else None)
print("I aligned with tokens:", len(tokens) == len(i_labels) if i_labels else None)
print("O aligned with tokens:", len(tokens) == len(o_labels) if o_labels else None)

print(f"\nParticipant labels of token file {filename}:")
print(p_labels[:21] if p_labels else None)

print(120 * "*")

print(f"Intervention labels of token file {filename}:")
print(i_labels[:21] if i_labels else None)

print(120 * "*")

print(f"Outcome labels of token file {filename}:")
print(o_labels[:21] if o_labels else None)


Matched sample files:
Token: C:/Users/vikra/OneDrive/Desktop/Study_Board/Bristol/SEM 2/Introduction to AI and Text Analytics/AITA Coursework/EBM-NLP-master/ebm_nlp_2_00/ebm_nlp_2_00/documents/8568534.tokens
Participant: C:/Users/vikra/OneDrive/Desktop/Study_Board/Bristol/SEM 2/Introduction to AI and Text Analytics/AITA Coursework/EBM-NLP-master/ebm_nlp_2_00/ebm_nlp_2_00/annotations/aggregated/hierarchical_labels/participants/train/8568534.AGGREGATED.ann
Intervention: C:/Users/vikra/OneDrive/Desktop/Study_Board/Bristol/SEM 2/Introduction to AI and Text Analytics/AITA Coursework/EBM-NLP-master/ebm_nlp_2_00/ebm_nlp_2_00/annotations/aggregated/hierarchical_labels/interventions/train/8568534.AGGREGATED.ann
Outcome: C:/Users/vikra/OneDrive/Desktop/Study_Board/Bristol/SEM 2/Introduction to AI and Text Analytics/AITA Coursework/EBM-NLP-master/ebm_nlp_2_00/ebm_nlp_2_00/annotations/aggregated/hierarchical_labels/outcomes/train/8568534.AGGREGATED.ann

Length checks:
P aligned with tokens: True
I

In [27]:
# Create a lookup from document ID to token file path
# This allows fast matching between token files and annotation files.

token_map = {
    f.split("/")[-1].replace(".tokens", ""): f
    for f in token_files
}

# Cache token files after first read to avoid repeated disk access
token_cache = {}

def read_tokens_cached(token_file):
    """
    Reads tokens from a file and stores them in a cache.
    This avoids re-reading the same token file multiple times,
    which improves efficiency when loading multiple entity types.
    """
    if token_file not in token_cache:
        token_cache[token_file] = read_lines_from_file(token_file)
    return token_cache[token_file]

def load_entity_from_folder(token_map, entity_files):
    """
    Loads token-label pairs for a given entity type.

    Parameters:
    - token_map: dictionary mapping document IDs to token file paths
    - entity_files: list of annotation files for one entity type

    Returns:
    - A list of document dictionaries, where each dictionary contains:
        - doc_id
        - tokens
        - labels

    The function matches annotation files with token files using document IDs,
    reads both sequences, checks alignment, and skips invalid or missing files.
    """
    data = []

    entity_map = {
        f.split("/")[-1].replace(".AGGREGATED.ann", ""): f
        for f in entity_files
    }

    for doc_id, ann_file in entity_map.items():
        token_file = token_map.get(doc_id)

        if token_file is None:
            continue

        tokens = read_tokens_cached(token_file)
        labels = read_lines_from_file(ann_file, as_int=True)

        if len(tokens) != len(labels):
            continue

        data.append({
            "doc_id": doc_id,
            "tokens": tokens,
            "labels": labels
        })

    return data

In [28]:
# sanity check
participants_train_data = load_entity_from_folder(token_map, participant_train_files)
participants_test_data = load_entity_from_folder(token_map, participant_test_files)

interventions_train_data = load_entity_from_folder(token_map, intervention_train_files)
interventions_test_data = load_entity_from_folder(token_map, intervention_test_files)

outcomes_train_data = load_entity_from_folder(token_map, outcome_train_files)
outcomes_test_data = load_entity_from_folder(token_map, outcome_test_files)

print("\nLoaded Docs:")
print("Participants train:", len(participants_train_data))
print("Participants test:", len(participants_test_data))
print("Interventions train:", len(interventions_train_data))
print("Interventions test:", len(interventions_test_data))
print("Outcomes train:", len(outcomes_train_data))
print("Outcomes test:", len(outcomes_test_data))



Loaded Docs:
Participants train: 4609
Participants test: 191
Interventions train: 4746
Interventions test: 191
Outcomes train: 4681
Outcomes test: 191


In [35]:
# sanity check on one file by fwtching it
if participants_train_data:
    sample = random.choice(participants_train_data)

    print("\nSample Doc ID:", sample["doc_id"])
    print("Tokens count:", len(sample["tokens"]))
    print("Labels count:", len(sample["labels"]))
    print("Aligned:", len(sample["tokens"]) == len(sample["labels"]))

    print("\nFirst 10 token-label pairs:")
    for t, l in zip(sample["tokens"][:10], sample["labels"][:10]):
        print(f"{t:20} -> {l}")


Sample Doc ID: 20620365
Tokens count: 393
Labels count: 393
Aligned: True

First 10 token-label pairs:
Translucency         -> 0
of                   -> 0
zirconia             -> 4
copings              -> 4
made                 -> 0
with                 -> 0
different            -> 0
CAD/CAM              -> 0
systems              -> 0
.                    -> 0


In [30]:
# acquiring true unique annotations
print("Participants unique labels:", sorted(set(x for doc in participants_train_data for x in doc["labels"])))
print("Interventions unique labels:", sorted(set(x for doc in interventions_train_data for x in doc["labels"])))
print("Outcomes unique labels:", sorted(set(x for doc in outcomes_train_data for x in doc["labels"])))

Participants unique labels: [0, 1, 2, 3, 4]
Interventions unique labels: [0, 1, 2, 3, 4, 5, 6, 7]
Outcomes unique labels: [0, 1, 2, 3, 4, 5, 6]


----

## Section 4: Label Transformation (RAW → BIO)

The EBM-NLP dataset provides token-level annotations in numeric form.  
However, for sequence tagging and span extraction, these labels need to be converted into **BIO format**:

- **B**: beginning of an entity span  
- **I**: inside an entity span  
- **O**: outside any entity span  

In this section, raw numeric labels are transformed into BIO tags by:
- assigning **O** to non-entity tokens (`0`)
- assigning **B** when a new entity begins
- assigning **I** when the entity continues across consecutive tokens

This conversion provides a more suitable representation for token-level classification and later span reconstruction.

In [36]:
def raw_labels_to_bio(raw_labels):
    """
    converts raw labels into BIO tags.

    Parameters:
    - raw_labels: list of integer labels from the dataset

    Returns:
    - list of BIO tags ('B', 'I', 'O')

    - 0 is treated as non-entity → O
    - a non-zero label that starts a new region → B
    - a non-zero label continuing the same region → I
    """
    
    bio_labels = []
    prev_label = 0

    for label in raw_labels:
        if label == 0:
            bio_labels.append("O")
        else:
            if label != prev_label:
                bio_labels.append("B")
            else:
                bio_labels.append("I")

        prev_label = label

    return bio_labels

In [46]:
# sanity checking on outcomes annotations ....
sample = random.choice(outcomes_train_data)

tokens = sample["tokens"]
raw_labels = sample["labels"]

bio_labels = raw_labels_to_bio(raw_labels)

In [47]:
print("Doc ID:", sample["doc_id"])
print("\nToken | Raw | BIO\n")

for t, r, b in zip(tokens[:30], raw_labels[:30], bio_labels[:30]):
    print(f"{t:20} {r:5} {b}")

Doc ID: 6189335

Token | Raw | BIO

Effect                   0 O
of                       0 O
betamethasone            0 O
valerate                 0 O
on                       0 O
the                      0 O
normal                   0 O
human                    1 B
facial                   1 I
skin                     1 I
flora                    1 I
.                        1 I
Eighteen                 0 O
volunteers               0 O
were                     0 O
randomly                 0 O
divided                  0 O
into                     0 O
two                      0 O
groups                   0 O
and                      0 O
allocated                0 O
either                   0 O
an                       0 O
active                   0 O
corticosteroid           0 O
preparation              0 O
(                        0 O
Betamethasone            0 O
valerate                 0 O


In [48]:
# adding BIO labels to all data ...

for doc in participants_train_data:
    doc["bio_labels"] = raw_labels_to_bio(doc["labels"])

for doc in participants_test_data:
    doc["bio_labels"] = raw_labels_to_bio(doc["labels"])

for doc in interventions_train_data:
    doc["bio_labels"] = raw_labels_to_bio(doc["labels"])

for doc in interventions_test_data:
    doc["bio_labels"] = raw_labels_to_bio(doc["labels"])

for doc in outcomes_train_data:
    doc["bio_labels"] = raw_labels_to_bio(doc["labels"])

for doc in outcomes_test_data:
    doc["bio_labels"] = raw_labels_to_bio(doc["labels"])

In [49]:
def bio_to_spans(tokens, bio_labels):
    """
    Converts BIO-tagged tokens into contiguous text spans.

    Parameters:
    - tokens: list of tokens in a document
    - bio_labels: corresponding BIO labels (B, I, O)

    Returns:
    - A list of extracted spans (phrases)

    The function groups consecutive B and I labels into complete entity spans,
    allowing conversion from token-level predictions to human-readable text.
    """
    spans = []
    current_span = []

    for token, label in zip(tokens, bio_labels):
        if label == "B":
            if current_span:
                spans.append(" ".join(current_span))
            current_span = [token]

        elif label == "I":
            if current_span:
                current_span.append(token)
            else:
                # handle broken BIO (I without B)
                current_span = [token]

        else:  # O
            if current_span:
                spans.append(" ".join(current_span))
                current_span = []

    if current_span:
        spans.append(" ".join(current_span))

    return spans

In [53]:
# sanity check on interventions sample file
sample = random.choice(interventions_train_data)

true_spans = bio_to_spans(sample["tokens"], sample["bio_labels"])

print("Doc ID:", sample["doc_id"])
print("\nExtracted spans:")
for s in true_spans[:10]:
    print("-", s)

Doc ID: 17003665

Extracted spans:
- Atomoxetine
- atomoxetine ( ATX )
- ATX
- placebo
- methylphenidate


---

## Section 5: Feature Engineering for Token Classification

To train a machine learning model for BIO tagging, token-level features are extracted from the text.

Each token is represented using a set of lexical and contextual features, including:
- Lowercased token form
- Capitalisation patterns
- Token length and digit information
- Previous and next token context

These features provide local context around each token, enabling the model to learn patterns associated with entity boundaries.

This representation is later converted into numerical form using a DictVectorizer before training the classification model.

In [54]:
# token features function

def token_features(tokens, i):
    """
    Extracts features for a token and its local context.

    Each token is represented using:
    - lexical features (case, length, digit information)
    - contextual features (previous and next tokens)

    This allows the model to use surrounding words to better identify
    entity boundaries in BIO tagging.
    """
    token = tokens[i]

    features = {
        "token.lower": token.lower(),
        "token.isupper": token.isupper(),
        "token.istitle": token.istitle(),
        "token.isdigit": token.isdigit(),
        "token.len": len(token),
    }

    # previous token
    if i > 0:
        prev_token = tokens[i - 1]
        features.update({
            "prev.lower": prev_token.lower(),
            "prev.istitle": prev_token.istitle(),
            "prev.isupper": prev_token.isupper(),
        })
    else:
        features["BOS"] = True   # beginning of sequence

    # next token
    if i < len(tokens) - 1:
        next_token = tokens[i + 1]
        features.update({
            "next.lower": next_token.lower(),
            "next.istitle": next_token.istitle(),
            "next.isupper": next_token.isupper(),
        })
    else:
        features["EOS"] = True   # end of sequence

    return features

---

## Section 6: End to End Formulation

Now we are exploring the impact of different training regimes on extraction performance. Specifically, the following strategies are compared:

- **Zero-shot**: Rule-based extraction using predefined cue words (no training)
- **Few-shot**: Supervised learning using a small subset of training data (100 documents)
- **Full-train**: Supervised learning using the complete training dataset

### Hypothesis
- Zero-shot will provide a weak baseline due to lack of learning
- Few-shot will capture useful patterns with limited supervision
- Full-train will improve recall but may introduce noise and false positives

---

### 6.1 Zero-shot Approach

In the zero-shot setting, the model is not trained on labelled data.
Instead, predictions are made using predefined heuristics or simple rules.

### Motivation

This approach tests whether the task can be solved without supervision.

### Expectation

We expect low performance, as no learning from data occurs.

### Importance

This serves as a baseline to measure the benefit of training.

---
## Zero-Shot Baseline

As a baseline approach, a zero-shot extraction strategy is implemented using simple rule-based heuristics.  
This approach does not involve any model training and relies entirely on predefined cue words associated with each entity type.

For each token, if it matches a set of manually defined keywords (e.g., "patients", "treatment", "effect"), it is labelled as part of an entity. BIO tags are then assigned based on consecutive matches.

### Motivation
The purpose of this approach is to establish a lower-bound performance for the task. Since no learning is involved, it highlights how much can be achieved using only domain intuition and simple rules.

This baseline provides a reference point for evaluating the effectiveness of supervised approaches (few-shot and full-train).

In [55]:
# cue word dictionaries
#For the zero-shot approach, a set of cue words is manually defined for each entity type:

participant_cues = {
    "patients", "patient", "subjects", "subject", "adults", "adult",
    "children", "child", "women", "woman", "men", "male", "female",
    "participants", "volunteers", "infants", "elderly"
}

intervention_cues = {
    "treatment", "therapy", "drug", "drugs", "dose", "doses", "placebo",
    "surgery", "surgical", "aspirin", "ibuprofen", "ranitidine",
    "lansoprazole", "methotrexate", "intervention", "regimen"
}

outcome_cues = {
    "pain", "relief", "mortality", "survival", "response", "improvement",
    "reduction", "outcome", "outcomes", "recovery", "bleeding", "cure",
    "healing", "efficacy", "safety"
}

In [72]:
import string

def clean_token(token):
    """
    Normalises a token by converting it to lowercase
    and removing surrounding punctuation.

    This helps improve matching between raw tokens
    and manually defined cue words.
    """
    return token.lower().strip(string.punctuation)

In [73]:
def predict_bio_zero_shot(tokens, cue_words, window=2):
    """
    Predicts BIO labels using a simple zero-shot heuristic.

    Parameters:
    - tokens: list of tokens in a document
    - cue_words: set of manually selected keywords for one entity type
    - window: number of following tokens to include as part of the span

    Returns:
    - A list of predicted BIO labels ('B', 'I', 'O')

    Logic:
    - if a token matches a cue word, mark it as the beginning of a span ('B')
    - extend the span to a small number of following tokens ('I')
    - all other tokens remain outside ('O')
    """
    pred_labels = ["O"] * len(tokens)

    for i, token in enumerate(tokens):
        tok = clean_token(token)

        if tok in cue_words:
            pred_labels[i] = "B"

            # extend to next few tokens if they are not punctuation
            for j in range(i + 1, min(i + 1 + window, len(tokens))):
                next_tok = clean_token(tokens[j])
                if next_tok and next_tok not in cue_words:
                    if tokens[j] not in string.punctuation:
                        pred_labels[j] = "I"
                else:
                    break

    return pred_labels

In [80]:
# Predict BIO labels using cue words: spans start at matched tokens (B)
# and extend to nearby tokens (I) using a small context window.

sample = random.choice(participants_test_data)

pred_bio = predict_bio_zero_shot(sample["tokens"], participant_cues)
pred_spans = bio_to_spans(sample["tokens"], pred_bio)

print("Doc ID:", sample["doc_id"])
print("\nPredicted spans:")
for s in pred_spans[:10]:
    print("-", s)

Doc ID: 8726604

Predicted spans:
- women undergoing elective
- women were randomized
- patients were oriented


In [81]:
gold_spans = bio_to_spans(sample["tokens"], sample["bio_labels"])

print("\nGold spans:")
for s in gold_spans[:10]:
    print("-", s)

print("\nPredicted spans:")
for s in pred_spans[:10]:
    print("-", s)


Gold spans:
- outpatient surgery .
- 47
- healthy
- women
- undergoing elective ambulatory surgery
- women

Predicted spans:
- women undergoing elective
- women were randomized
- patients were oriented


In [85]:
def spans_to_set(spans):
    return set(s.lower().strip() for s in spans)

def evaluate_doc_spans(gold_spans, pred_spans):
    """
    Computes span-level TP, FP, FN for a single document.

    Compares predicted spans with gold spans after normalisation:
    - TP: correctly predicted spans
    - FP: extra predicted spans
    - FN: missed gold spans
    """
    
    gold_set = spans_to_set(gold_spans)
    pred_set = spans_to_set(pred_spans)

    tp = len(gold_set & pred_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)

    return tp, fp, fn

In [86]:
def evaluate_zero_shot_dataset(docs, cue_words):
    """
    Evaluates zero-shot extraction performance over a dataset.

    For each document:
    - generates gold spans from true BIO labels
    - predicts BIO labels using zero-shot heuristics
    - converts predictions to spans
    - computes TP, FP, FN

    Aggregates results across all documents and returns
    overall precision, recall, and F1-score.
    """
    total_tp, total_fp, total_fn = 0, 0, 0

    for doc in docs:
        gold_spans = bio_to_spans(doc["tokens"], doc["bio_labels"])
        pred_bio = predict_bio_zero_shot(doc["tokens"], cue_words)
        pred_spans = bio_to_spans(doc["tokens"], pred_bio)

        tp, fp, fn = evaluate_doc_spans(gold_spans, pred_spans)

        total_tp += tp
        total_fp += fp
        total_fn += fn

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": total_tp,
        "fp": total_fp,
        "fn": total_fn
    }

In [87]:
# implementing zero shot  ...... 
p_zero_results = evaluate_zero_shot_dataset(participants_test_data, participant_cues)
i_zero_results = evaluate_zero_shot_dataset(interventions_test_data, intervention_cues)
o_zero_results = evaluate_zero_shot_dataset(outcomes_test_data, outcome_cues)

print("Participants zero-shot:", p_zero_results)
print("Interventions zero-shot:", i_zero_results)
print("Outcomes zero-shot:", o_zero_results)

Participants zero-shot: {'precision': 0.04432855280312908, 'recall': 0.04176904176904177, 'f1': 0.043010752688172046, 'tp': 34, 'fp': 733, 'fn': 780}
Interventions zero-shot: {'precision': 0.054673721340388004, 'recall': 0.03822441430332922, 'f1': 0.04499274310595065, 'tp': 31, 'fp': 536, 'fn': 780}
Outcomes zero-shot: {'precision': 0.07446808510638298, 'recall': 0.02241793434747798, 'f1': 0.03446153846153846, 'tp': 28, 'fp': 348, 'fn': 1221}


In [88]:
# structuring overall results in dataframe

import pandas as pd

zero_results_df = pd.DataFrame([
    {"entity": "Participants", "strategy": "zero-shot", **p_zero_results},
    {"entity": "Interventions", "strategy": "zero-shot", **i_zero_results},
    {"entity": "Outcomes", "strategy": "zero-shot", **o_zero_results},
])

zero_results_df

,entity,strategy,precision,recall,f1,tp,fp,fn
0,Participants,zero-shot,0.044329,0.041769,0.043011,34,733,780
1,Interventions,zero-shot,0.054674,0.038224,0.044993,31,536,780
2,Outcomes,zero-shot,0.074468,0.022418,0.034462,28,348,1221


---

### Zero-Shot Results and Observations

The zero-shot approach shows very poor performance across all entity types.

- Participants: F1 ≈ 0.04  
- Interventions: F1 ≈ 0.04  
- Outcomes: F1 ≈ 0.03  

### Key Observations

- **Very low precision** indicates that many predicted spans are incorrect (high false positives).  
- **Very low recall** shows that the model fails to capture most of the true spans (high false negatives).  
- The model often extracts **incomplete or noisy spans**, such as partial phrases or irrelevant tokens.  

### Interpretation

The zero-shot approach relies on simple cue-word matching and does not learn from data. As a result:

- it cannot capture contextual meaning  
- it fails to identify correct span boundaries  
- it struggles with sentences containing multiple or complex entity references  

### Conclusion

These results demonstrate that zero-shot methods are insufficient for this task.  
They serve as a weak baseline and highlight the need for supervised learning approaches (few-shot and full training) to achieve meaningful extraction performance.

------

## 6.2 Few-shot Baseline 

The few-shot approach introduces a limited amount of labelled data to train the model, allowing it to learn basic patterns for entity extraction.
Unlike the zero-shot setting, where predictions are based solely on predefined heuristics, the few-shot model is trained on a small subset of annotated examples. This enables the model to move beyond simple keyword matching and begin capturing local contextual patterns in the text.

### Motivation

The poor performance of the zero-shot approach highlights the limitations of rule-based methods. In particular, the model struggles with:

- identifying correct entity boundaries  
- handling contextual variation  
- distinguishing between similar phrases with different meanings  

By providing a small amount of training data, the few-shot approach aims to improve generalisation while still operating under limited supervision.

### Expected Outcome

With minimal training, we expect:

- improved precision due to better pattern recognition  
- moderate improvement in recall  
- more coherent span extraction compared to zero-shot  

However, performance may still be limited due to the small size of the training data.

In [89]:
def sample_few_shot_docs(docs, n=100, seed=42):
    """
    Randomly samples a subset of documents for few-shot training.

    Parameters:
    - docs: full training dataset
    - n: number of documents to sample
    - seed: random seed for reproducibility

    Returns:
    - A list of sampled documents (size ≤ n)
    """
    
    random.seed(seed)
    return random.sample(docs, min(n, len(docs)))

### Preparing Token-Level Training Data

To train the classification model, each document is converted into token-level training instances.

For every token in each document:
- a feature dictionary is generated using the `token_features` function
- the corresponding BIO label is assigned as the target

This results in a flattened dataset where each token becomes an independent training example, enabling standard classification models to be applied to sequence tagging.

In [91]:
def prepare_token_classification_data(docs):
    """
    Converts document-level data into token-level training examples.

    Parameters:
    - docs: list of documents, each containing tokens and BIO labels

    Returns:
    - X: list of feature dictionaries (one per token)
    - y: list of corresponding BIO labels

    Each token in every document is treated as an independent
    training instance, allowing standard classifiers to be used
    for sequence tagging.
    """
    
    X = []
    y = []

    for doc in docs:
        tokens = doc["tokens"]
        bio_labels = doc["bio_labels"]

        for i in range(len(tokens)):
            X.append(token_features(tokens, i))
            y.append(bio_labels[i])

    return X, y

In [94]:
def sample_few_shot_docs(docs, n=100, seed=42):
    random.seed(seed)
    return random.sample(docs, min(n, len(docs)))

---

This function runs a few-shot experiment using a limited number of training documents (n_shot=100).
It samples 100 documents from the training set, extracts token-level features, and trains a
Logistic Regression model for BIO tagging. The model is then evaluated on the full test set,
enabling comparison of performance under constrained supervision.

In [98]:
### Few-Shot Training and Evaluation

# The following function implements the complete few-shot pipeline in a reusable form.
# A small subset of the training data is first sampled to simulate a low-resource learning setting. Token-level features are then extracted from both the sampled training set and the full test set. These features are converted into numerical vectors using a `DictVectorizer`, and a Logistic Regression model is trained to predict BIO labels.
# The trained model is evaluated on the full test set using token-level precision, recall, and F1-score. This setup enables direct comparison between few-shot, zero-shot, and full-training strategies.

def run_few_shot_experiment(train_data, test_data, n_shot=100, seed=42):
    """
    Runs the complete few-shot training and evaluation pipeline.

    Parameters:
    - train_data: full training dataset
    - test_data: full test dataset
    - n_shot: number of training documents to sample
    - seed: random seed for reproducibility

    Returns:
    - trained model
    - fitted vectorizer
    - predicted BIO labels on the test set
    - true BIO labels for the test set

    The function samples a small subset of training documents,
    converts tokens into feature dictionaries, vectorises them,
    trains a Logistic Regression model, and evaluates token-level performance.
    """
    few_shot_train = sample_few_shot_docs(train_data, n=n_shot, seed=seed)

    X_train_dict, y_train = prepare_token_classification_data(few_shot_train)
    X_test_dict, y_test = prepare_token_classification_data(test_data)

    vectorizer = DictVectorizer(sparse=True)
    X_train = vectorizer.fit_transform(X_train_dict)
    X_test = vectorizer.transform(X_test_dict)

    model = LogisticRegression(max_iter=800, class_weight="balanced")
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print(classification_report(y_test, y_pred))

    return model, vectorizer, y_pred, y_test

In [99]:
# implementing few-shot startegy to P,I,Os

print("=== Participants Few-Shot ===")
p_few_model, p_few_vectorizer, p_few_pred, p_few_true = run_few_shot_experiment(
    participants_train_data, participants_test_data, n_shot=100, seed=42
)

print("=== Interventions Few-Shot ===")
i_few_model, i_few_vectorizer, i_few_pred, i_few_true = run_few_shot_experiment(
    interventions_train_data, interventions_test_data, n_shot=100, seed=42
)

print("=== Outcomes Few-Shot ===")
o_few_model, o_few_vectorizer, o_few_pred, o_few_true = run_few_shot_experiment(
    outcomes_train_data, outcomes_test_data, n_shot=100, seed=42
)

=== Participants Few-Shot ===
              precision    recall  f1-score   support

           B       0.27      0.51      0.35       869
           I       0.25      0.31      0.28      2591
           O       0.96      0.93      0.95     48324

    accuracy                           0.89     51784
   macro avg       0.49      0.58      0.52     51784
weighted avg       0.91      0.89      0.90     51784

=== Interventions Few-Shot ===
              precision    recall  f1-score   support

           B       0.31      0.38      0.34      1534
           I       0.16      0.40      0.23      1382
           O       0.96      0.92      0.94     48868

    accuracy                           0.89     51784
   macro avg       0.48      0.56      0.50     51784
weighted avg       0.92      0.89      0.90     51784

=== Outcomes Few-Shot ===
              precision    recall  f1-score   support

           B       0.22      0.43      0.29      1442
           I       0.27      0.41      0.3

---

## Few-Shot Results (Token-Level)

The few-shot model (trained on 100 documents) achieves moderate performance across all entity types, with macro F1 scores around **0.50–0.52** and overall accuracy between **0.85–0.89**.

Performance is highly influenced by class imbalance:
- The **“O” (non-entity)** class shows very high performance (F1 ≈ **0.92–0.95**)
- The **“B” and “I” (entity)** classes perform significantly worse (F1 ≈ **0.23–0.35**)

### Key Observations
- The model is able to **identify potential entity regions** (reasonable recall)
- However, it struggles with **precise boundary detection**, especially for **I labels**
- This results in **fragmented and incomplete spans**

### Interpretation
Although token-level accuracy is high, this is largely due to the dominance of the “O” class. The relatively lower performance on B and I labels indicates that the model has limited ability to capture complete entity spans under few-shot supervision.

---

## 6.3 Full Training BIO prediction

In this section, the model is trained using the full training dataset instead of a limited subset.

This represents a fully supervised learning setting, where the model has access to all available annotated data. The goal is to evaluate whether increasing the amount of training data improves extraction performance.

### Hypothesis
- Recall is expected to improve, as the model has seen more examples
- However, precision may decrease due to over-prediction and noise

In [100]:
def run_full_train_experiment(train_data, test_data):
    """
    Runs the complete few-shot training and evaluation pipeline.

    Parameters:
    - train_data: full training dataset
    - test_data: full test dataset
    - n_shot: number of training documents to sample
    - seed: random seed for reproducibility

    Returns:
    - trained model
    - fitted vectorizer
    - predicted BIO labels on the test set
    - true BIO labels for the test set

    The function samples a small subset of training documents,
    converts tokens into feature dictionaries, vectorises them,
    trains a Logistic Regression model, and evaluates token-level performance.
    """
    
    X_train_dict, y_train = prepare_token_classification_data(train_data)
    X_test_dict, y_test = prepare_token_classification_data(test_data)

    vectorizer = DictVectorizer(sparse=True)
    X_train = vectorizer.fit_transform(X_train_dict)
    X_test = vectorizer.transform(X_test_dict)

    model = LogisticRegression(max_iter=800, class_weight="balanced")
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print(classification_report(y_test, y_pred))

    return model, vectorizer, y_pred, y_test

In [101]:
print("=== Participants Full-Train ===")
p_full_model, p_full_vectorizer, p_full_pred, p_full_true = run_full_train_experiment(
    participants_train_data, participants_test_data
)

print("=== Interventions Full-Train ===")
i_full_model, i_full_vectorizer, i_full_pred, i_full_true = run_full_train_experiment(
    interventions_train_data, interventions_test_data
)

print("=== Outcomes Full-Train ===")
o_full_model, o_full_vectorizer, o_full_pred, o_full_true = run_full_train_experiment(
    outcomes_train_data, outcomes_test_data
)

=== Participants Full-Train ===


C:\Users\vikra\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


              precision    recall  f1-score   support

           B       0.19      0.78      0.31       869
           I       0.21      0.59      0.31      2591
           O       0.98      0.84      0.90     48324

    accuracy                           0.82     51784
   macro avg       0.46      0.73      0.51     51784
weighted avg       0.93      0.82      0.86     51784

=== Interventions Full-Train ===


C:\Users\vikra\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


              precision    recall  f1-score   support

           B       0.26      0.83      0.40      1534
           I       0.13      0.82      0.22      1382
           O       0.99      0.77      0.87     48868

    accuracy                           0.77     51784
   macro avg       0.46      0.81      0.49     51784
weighted avg       0.95      0.77      0.83     51784

=== Outcomes Full-Train ===


C:\Users\vikra\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


              precision    recall  f1-score   support

           B       0.19      0.80      0.31      1442
           I       0.25      0.76      0.37      3180
           O       0.99      0.75      0.85     47162

    accuracy                           0.76     51784
   macro avg       0.47      0.77      0.51     51784
weighted avg       0.92      0.76      0.81     51784



---

## Full-Train Results (Token-Level)

The full-train model, trained on the complete dataset, shows a significant shift in behaviour compared to the few-shot setting. While overall accuracy decreases slightly (≈ 0.76–0.82), recall for entity classes improves substantially.

### Performance characteristics:

The “O” (non-entity) class remains strong, though slightly reduced (F1 ≈ 0.85–0.90)
The “B” and “I” (entity) classes show high recall (≈ 0.59–0.83), but low precision (≈ 0.13–0.26)
Key Observations
The model becomes highly recall-oriented, capturing a large proportion of entity tokens
However, precision drops significantly, indicating over-prediction of entity spans
The model frequently extends spans incorrectly, especially for I labels, leading to noisy boundaries
Compared to few-shot, the model is less conservative and more aggressive in tagging entities

### Interpretation

The increase in training data enables the model to learn broader patterns, improving its ability to detect entity regions. However, due to limited sequence modelling and reliance on local features, the model struggles with precise boundary detection.

As a result, while recall improves, the model produces more false positives, leading to noisy and fragmented spans. Despite better coverage, overall F1 scores remain comparable to the few-shot setting, highlighting a trade-off between precision and recall.

In [102]:
## combining all results in dataframe

import pandas as pd

results = [
    # Zero-shot
    {"entity": "Participants", "strategy": "zero-shot", "precision": 0.0443, "recall": 0.0418, "f1": 0.0430},
    {"entity": "Interventions", "strategy": "zero-shot", "precision": 0.0547, "recall": 0.0382, "f1": 0.0450},
    {"entity": "Outcomes", "strategy": "zero-shot", "precision": 0.0745, "recall": 0.0224, "f1": 0.0345},

    # Few-shot
    {"entity": "Participants", "strategy": "few-shot", "precision": 0.27, "recall": 0.51, "f1": 0.35},
    {"entity": "Interventions", "strategy": "few-shot", "precision": 0.31, "recall": 0.38, "f1": 0.34},
    {"entity": "Outcomes", "strategy": "few-shot", "precision": 0.22, "recall": 0.43, "f1": 0.29},

    # Full-train
    {"entity": "Participants", "strategy": "full-train", "precision": 0.19, "recall": 0.78, "f1": 0.31},
    {"entity": "Interventions", "strategy": "full-train", "precision": 0.26, "recall": 0.83, "f1": 0.40},
    {"entity": "Outcomes", "strategy": "full-train", "precision": 0.19, "recall": 0.80, "f1": 0.31},
]

df = pd.DataFrame(results)

df.round(3)

,entity,strategy,precision,recall,f1
0,Participants,zero-shot,0.044,0.042,0.043
1,Interventions,zero-shot,0.055,0.038,0.045
2,Outcomes,zero-shot,0.074,0.022,0.034
3,Participants,few-shot,0.270,0.510,0.350
4,Interventions,few-shot,0.310,0.380,0.340
5,Outcomes,few-shot,0.220,0.430,0.290
6,Participants,full-train,0.190,0.780,0.310
7,Interventions,full-train,0.260,0.830,0.400
8,Outcomes,full-train,0.190,0.800,0.310


---

## Section 7: Span-Level Prediction and Evaluation

While token-level predictions provide insight into model behaviour, the primary objective of this task is to extract meaningful entity spans corresponding to Population, Intervention, and Outcome.

In this section, token-level BIO predictions are converted into contiguous text spans. These predicted spans are then compared against the gold (true) spans obtained from the dataset annotations.

Evaluation is performed at the span level using precision, recall, and F1-score, based on:
- True Positives (TP): correctly predicted spans
- False Positives (FP): extra predicted spans not present in the gold set
- False Negatives (FN): gold spans that were not predicted

This evaluation provides a more realistic measure of extraction quality, as it reflects the model’s ability to correctly identify complete entity phrases rather than individual tokens.

In [103]:
def normalize_spans(spans):
    return set(s.lower().strip() for s in spans)

In [104]:
def evaluate_spans(gold_spans, pred_spans):
    """
    Computes span-level True Positives, False Positives, and False Negatives.

    Parameters:
    - gold_spans: list of ground truth spans
    - pred_spans: list of predicted spans

    Returns:
    - tp: correctly predicted spans
    - fp: predicted spans not in gold
    - fn: gold spans missed by the model

    Spans are first normalised (lowercased and stripped) to ensure
    consistent comparison.
    """
    gold_set = normalize_spans(gold_spans)
    pred_set = normalize_spans(pred_spans)

    tp = len(gold_set & pred_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)

    return tp, fp, fn


In [105]:
def evaluate_span_level(docs, pred_bio_fn):
    """
    Evaluates span-level extraction performance over a dataset.

    For each document:
    - converts gold BIO labels into spans
    - generates predicted BIO labels using the provided prediction function
    - converts predicted BIO labels into spans
    - compares predicted and gold spans to compute TP, FP, FN

    Aggregates results across all documents and computes:
    - precision, recall, and F1-score

    Parameters:
    - docs: list of documents with tokens and gold BIO labels
    - pred_bio_fn: function that generates predicted BIO labels for a document

    Returns:
    - Dictionary containing precision, recall, F1, and TP/FP/FN counts
    """
    total_tp, total_fp, total_fn = 0, 0, 0

    for doc in docs:
        tokens = doc["tokens"]

        # gold spans
        gold_spans = bio_to_spans(tokens, doc["bio_labels"])

        # predicted BIO labels
        pred_bio = pred_bio_fn(doc)

        # predicted spans
        pred_spans = bio_to_spans(tokens, pred_bio)

        # compare
        tp, fp, fn = evaluate_spans(gold_spans, pred_spans)

        total_tp += tp
        total_fp += fp
        total_fn += fn

    precision = total_tp / (total_tp + total_fp) if total_tp + total_fp > 0 else 0
    recall = total_tp / (total_tp + total_fn) if total_tp + total_fn > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "TP": total_tp,
        "FP": total_fp,
        "FN": total_fn
    }

In [122]:
# -----------------------------------
# 7.1 ZERO-SHOT WRAPPER
# -----------------------------------

def zero_shot_pred_fn_factory(cue_words):
    """
    Creates a prediction function for zero-shot BIO tagging.
    """
    def pred_fn(doc):
        return predict_bio_zero_shot(doc["tokens"], cue_words)
    return pred_fn


In [124]:
# -----------------------------------
# 7.2. FEW/FULL MODEL WRAPPER
# -----------------------------------

def model_pred_fn_factory(model, vectorizer):
    """
    Creates a prediction function using a trained model and vectorizer.

    Parameters:
    - model: trained classification model (e.g., Logistic Regression)
    - vectorizer: fitted DictVectorizer used to transform features

    Returns:
    - A function that takes a document and returns predicted BIO labels

    The function extracts token-level features, transforms them into
    numerical vectors, and uses the trained model to predict BIO labels.
    This allows learned models (few-shot or full-train) to be used
    within a generic span-level evaluation pipeline.
    """
    def pred_fn(doc):
        tokens = doc["tokens"]
        features = [token_features(tokens, i) for i in range(len(tokens))]
        X = vectorizer.transform(features)
        return model.predict(X)
    return pred_fn

In [125]:
# -----------------------------------
# RUNNING SPAN-LEVEL EVALUATION
# -----------------------------------

# Zero-shot
p_zero = evaluate_span_level(
    participants_test_data,
    zero_shot_pred_fn_factory(participant_cues)
)

i_zero = evaluate_span_level(
    interventions_test_data,
    zero_shot_pred_fn_factory(intervention_cues)
)

o_zero = evaluate_span_level(
    outcomes_test_data,
    zero_shot_pred_fn_factory(outcome_cues)
)


In [126]:
# Few-shot
p_few = evaluate_span_level(
    participants_test_data,
    model_pred_fn_factory(p_few_model, p_few_vectorizer)
)

i_few = evaluate_span_level(
    interventions_test_data,
    model_pred_fn_factory(i_few_model, i_few_vectorizer)
)

o_few = evaluate_span_level(
    outcomes_test_data,
    model_pred_fn_factory(o_few_model, o_few_vectorizer)
)


In [127]:

# Full-train
p_full = evaluate_span_level(
    participants_test_data,
    model_pred_fn_factory(p_full_model, p_full_vectorizer)
)

i_full = evaluate_span_level(
    interventions_test_data,
    model_pred_fn_factory(i_full_model, i_full_vectorizer)
)

o_full = evaluate_span_level(
    outcomes_test_data,
    model_pred_fn_factory(o_full_model, o_full_vectorizer)
)


In [128]:
# -----------------------------------
# 6. FINAL RESULTS TABLE
# -----------------------------------

results_df = pd.DataFrame([
    {"entity": "Participants", "strategy": "zero-shot", **p_zero},
    {"entity": "Participants", "strategy": "few-shot", **p_few},
    {"entity": "Participants", "strategy": "full-train", **p_full},

    {"entity": "Interventions", "strategy": "zero-shot", **i_zero},
    {"entity": "Interventions", "strategy": "few-shot", **i_few},
    {"entity": "Interventions", "strategy": "full-train", **i_full},

    {"entity": "Outcomes", "strategy": "zero-shot", **o_zero},
    {"entity": "Outcomes", "strategy": "few-shot", **o_few},
    {"entity": "Outcomes", "strategy": "full-train", **o_full},
])

print(results_df)

          entity    strategy  precision    recall        f1   TP    FP    FN
0   Participants   zero-shot   0.044329  0.041769  0.043011   34   733   780
1   Participants    few-shot   0.075000  0.232187  0.113377  189  2331   625
2   Participants  full-train   0.044066  0.264128  0.075531  215  4664   599
3  Interventions   zero-shot   0.054674  0.038224  0.044993   31   536   780
4  Interventions    few-shot   0.096510  0.334155  0.149765  271  2537   540
5  Interventions  full-train   0.068415  0.485820  0.119939  394  5365   417
6       Outcomes   zero-shot   0.074468  0.022418  0.034462   28   348  1221
7       Outcomes    few-shot   0.086112  0.277022  0.131384  346  3672   903
8       Outcomes  full-train   0.064714  0.355484  0.109494  444  6417   805


In [129]:
results_df.round(3)

,entity,strategy,precision,recall,f1,TP,FP,FN
0,Participants,zero-shot,0.044,0.042,0.043,34,733,780
1,Participants,few-shot,0.075,0.232,0.113,189,2331,625
2,Participants,full-train,0.044,0.264,0.076,215,4664,599
3,Interventions,zero-shot,0.055,0.038,0.045,31,536,780
4,Interventions,few-shot,0.097,0.334,0.150,271,2537,540
5,Interventions,full-train,0.068,0.486,0.120,394,5365,417
6,Outcomes,zero-shot,0.074,0.022,0.034,28,348,1221
7,Outcomes,few-shot,0.086,0.277,0.131,346,3672,903
8,Outcomes,full-train,0.065,0.355,0.109,444,6417,805


## Span-Level Results

Span-level evaluation reveals that the few-shot model consistently outperforms both the zero-shot and full-train approaches across all entity types. While the full-train model achieves higher recall, it suffers from a substantial increase in false positives, leading to lower precision and overall F1-score.

This indicates that although additional training data helps the model detect more entity regions, it also causes over-prediction and noisy span boundaries. In contrast, the few-shot model provides a better balance between precision and recall, resulting in cleaner and more accurate span extraction.

The results highlight a key limitation of the approach: the model struggles with exact span reconstruction, and even small boundary mismatches are penalised in span-level evaluation.

---

## Section 8: PICO Table Extraction

The final step of the pipeline involves converting extracted spans into a structured tabular format.

Using the predicted spans for each document, the extracted entities are organised into a table with the following fields:
- Population
- Intervention
- Outcome

Although the standard PICO framework includes a Comparator component, it is not explicitly modelled in this implementation. This is due to the complexity of reliably distinguishing comparator information using the current token-level approach and available annotations.

Each row in the resulting table corresponds to a document, where extracted spans are aggregated into their respective fields. This structured representation demonstrates how unstructured clinical text can be transformed into a format suitable for downstream tasks such as querying, summarisation, and evidence synthesis.

In [142]:
def get_pred_spans_dict(docs, pred_fn):
    """
    Generates predicted spans for each document using a BIO prediction function.
    Returns a dictionary: doc_id -> list of predicted spans.
    """
    out = {}
    for doc in docs:
        tokens = doc["tokens"]
        pred_bio = pred_fn(doc)
        spans = bio_to_spans(tokens, pred_bio)
        out[doc["doc_id"]] = spans
    return out


In [143]:
p_pred = get_pred_spans_dict(
    participants_test_data,
    model_pred_fn_factory(p_few_model, p_few_vectorizer)
)

i_pred = get_pred_spans_dict(
    interventions_test_data,
    model_pred_fn_factory(i_few_model, i_few_vectorizer)
)

o_pred = get_pred_spans_dict(
    outcomes_test_data,
    model_pred_fn_factory(o_few_model, o_few_vectorizer)
)

In [144]:
def clean_span_list(spans):
    """
    Cleans extracted spans by removing empty, punctuation-only,
    numeric-only, and duplicate spans.
    """
    cleaned = []
    seen = set()

    for s in spans:
        s = s.strip()
        s = re.sub(r"\s+", " ", s)

        if not s:
            continue
        if re.fullmatch(r"\W+", s):
            continue
        if re.fullmatch(r"\d+(\.\d+)?", s):
            continue
        if len(s.split()) == 1 and len(s) <= 2:
            continue

        key = s.lower()
        if key not in seen:
            seen.add(key)
            cleaned.append(s)

    return cleaned

In [145]:
def build_clean_pio_table(doc_ids, p_spans, i_spans, o_spans):
    """
    Builds a structured PIO table from predicted spans.
    Each row corresponds to one document.
    """
    rows = []

    for d in doc_ids:
        p_clean = clean_span_list(p_spans.get(d, []))
        i_clean = clean_span_list(i_spans.get(d, []))
        o_clean = clean_span_list(o_spans.get(d, []))

        rows.append({
            "doc_id": d,
            "Population": " ; ".join(p_clean),
            "Intervention": " ; ".join(i_clean),
            "Outcome": " ; ".join(o_clean),
        })

    return pd.DataFrame(rows)


In [146]:
doc_ids = [d["doc_id"] for d in participants_test_data]

clean_pio_df = build_clean_pio_table(doc_ids, p_pred, i_pred, o_pred)
clean_pio_df.head(10)

,doc_id,Population,Intervention,Outcome
0,10070173,allergic rhinitis ; Study ; allergic rhinitis ...,Turbuhaler ; rhinitis ; SAR ; preferences ; sp...,seasonal allergic rhinitis ; effect ; Secondar...
1,10390665,post-operative ; children ; with or without ; ...,post-operative ; granisetron ; tonsillectomy ;...,Anti-emetic efficacy ; post-operative vomiting...
2,10475150,healthy volunteers and ; patients with ; secre...,somatostatin ; octreotide ) ; be involved ; me...,somatostatin ; regulation ; increased ; octreo...
3,10578479,premenopausal women ; protect ; women ; agains...,metabolism ; phytoestrogens ; hormone therapie...,soy ; phytoestrogens ; metabolism ; menstrual ...
4,10589810,loss ; 20 subjects ; a control group ; mean ; ...,steel ; therapy ( SPT ; periodontal ; therapy ...,Clinical effects ; conventional ; therapy ; ti...
5,10752495,patients with ; 38 patients with ; 20 controls...,glomerular ; arthritis ; endogenous ; ( Clcr )...,clearance from serum creatinine ; of glomerula...
6,10763172,Xylometazoline ; Dexpanthenol ; AND ; verum ; ...,xylometazoline ; combination of ; Dexpanthenol...,wound healing ; xylometazoline ; dexpanthenol ...
7,10764172,surgery . ; patients who underwent ; Forty pat...,Intrathecal ; abdominal surgery ; morphine ; i...,suppresses NK cell activity ; ( NK ) cell acti...
8,10912743,hypertensive patients . ; Project ; -- Hyperte...,aspirin ; Collaborative ; Project ; PPP ) ; ) ...,ambulatory blood pressure ; Nonsteroidal antii...
9,10934569,children ; with ; developmental disorder . ; Y...,intervention for ; parent training ; group ; N...,measures ; intelligence ; visual-spatial skill...


---

### Qualitative Analysis of Extracted PIO Table

The generated PIO table demonstrates that while the model is able to identify relevant entity-related tokens, the extracted spans are often fragmented, noisy, and semantically incomplete.

Common issues include:
- Inclusion of irrelevant or generic terms (e.g., “study”, “mean”, “loss”)
- Fragmented spans due to incorrect boundary detection
- Mixing of unrelated concepts within the same field
- Repetition of similar phrases

These limitations arise from the use of a token-level classification approach, which does not explicitly model full sentence context or semantic relationships between tokens.

As a result, although the system captures useful signals, the final structured output remains partially noisy and requires further refinement for practical use.

## Section 8: Conclusion

This study implemented an end-to-end pipeline for extracting structured PIO information from clinical trial abstracts. The results show that zero-shot methods are ineffective, while few-shot learning provides a strong improvement by capturing useful patterns with limited data.

Although full-train models achieve higher recall, they suffer from lower precision due to over-prediction, resulting in noisy outputs. Span-level evaluation highlights that the model struggles with accurate boundary detection, leading to fragmented entity extraction.

Overall, the approach demonstrates the importance of supervised learning but also reveals the limitations of token-level models in producing high-quality structured outputs. More advanced, context-aware methods are needed for reliable information extraction in real-world settings.